# Session 9 — CrewAI Advanced
## Focus: Production-quality CrewAI workflows
### Introduction to CrewAI Advanced Concepts

**Explanation:** CrewAI represents a significant evolution in multi-agent systems by providing a framework for creating sophisticated, collaborative AI agents that can handle complex workflows. Unlike simpler AI assistants, CrewAI enables the creation of specialized agents that work together in coordinated teams, each with distinct roles, tools, and responsibilities. This session focuses on moving beyond basic implementations to production-ready systems that incorporate advanced features like human-in-the-loop oversight, multimodal capabilities, and robust error handling. These advanced workflows are designed to handle real-world business scenarios where reliability, scalability, and maintainability are critical requirements.

## Installation & Requirements Setup

In [1]:
# Install required packages for CrewAI with OpenRouter
!pip install crewai openai python-dotenv requests beautifulsoup4 langchain

  Using cached crewai-0.193.2-py3-none-any.whl.metadata (35 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached chromadb-1.1.0-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached click-8.3.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached instructor-1.11.3-py3-none-any.whl.metadata (11 kB)
  Using cached json_repair-0.25.2-py3-none-any.whl.metadata (7.9 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached litellm-1.74.9-py3-none-any.whl.metadata (40 kB)
  Using cached onnxruntime-1.22.0-cp313-cp313-win_amd64.whl.metadata (5.0 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached opentelemetry_api-1.37.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setting up CrewAI

In [2]:
import os
import json
from datetime import datetime
from typing import Dict, List, Any, Optional
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure OpenRouter API
os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a"  #remove 'a' at the end to use this api key

**Explanation:** This installation and setup section establishes the foundation for our advanced CrewAI implementation. The required packages include crewai for the core multi-agent framework, openai for API interactions (configured to work with OpenRouter), and supporting libraries for web scraping, environment management, and data processing. The environment configuration ensures secure API key management and prepares the system for production deployment with proper credential handling.

## Building a Customer Support Crew with HITL Escalation

In [13]:
class CustomerSupportCrew:
    def __init__(self):
        self.llm_config = {
            "model": "meta-llama/llama-3.1-8b-instruct",
            "base_url": "https://openrouter.ai/api/v1",
            "api_key": "sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a" #remove 'a' at the end to use this api key
        }

        # Initialize specialized agents
        self.triage_agent = self._create_triage_agent()
        self.technical_agent = self._create_technical_agent()
        self.billing_agent = self._create_billing_agent()
        self.escalation_agent = self._create_escalation_agent()

        # Knowledge base simulation
        self.knowledge_base = self._load_knowledge_base()
        self.conversation_history = []

    def _load_knowledge_base(self) -> Dict:
        """Simulate a knowledge base with common solutions"""
        return {
            "common_issues": {
                "login_problems": "Clear browser cache or reset password using the forgot password feature",
                "payment_failed": "Check card details, ensure sufficient funds, or try alternative payment method",
                "api_errors": "Verify API key validity and check documentation for current rate limits",
                "slow_performance": "Clear cache, check internet connection, or try during off-peak hours"
            },
            "product_features": {
                "api_integration": "REST API with OAuth2 authentication, rate limiting, and comprehensive documentation",
                "billing": "Monthly subscriptions with automated invoicing, multiple payment methods, and prorated charges",
                "support": "24/7 email support with 4-hour response time SLA and dedicated account managers for enterprise"
            }
        }


**Explanation:** The CustomerSupportCrew class initializes a multi-agent system with specialized roles for handling different types of customer inquiries. The triage agent categorizes incoming requests, technical agents handle product issues, billing agents manage subscription concerns, and escalation agents facilitate human intervention when needed. The knowledge base provides contextual information that all agents can access, ensuring consistent and accurate responses across the team. The conversation history maintains context across interactions, enabling continuous learning and personalized support.

## Creating Specialized Support Agents

In [14]:
 class CustomerSupportCrew:
    def __init__(self):
        self.llm_config = {
            "model": "meta-llama/llama-3.1-8b-instruct",
            "base_url": "https://openrouter.ai/api/v1",
            "api_key": "sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a"  #remove 'a' at the end to use this api key
        }

        # Initialize specialized agents
        self.triage_agent = self._create_triage_agent()
        self.technical_agent = self._create_technical_agent()
        self.billing_agent = self._create_billing_agent()
        self.escalation_agent = self._create_escalation_agent()

        # Knowledge base simulation
        self.knowledge_base = self._load_knowledge_base()
        self.conversation_history = []

    def _load_knowledge_base(self) -> Dict:
        """Simulate a knowledge base with common solutions"""
        return {
            "common_issues": {
                "login_problems": "Clear browser cache or reset password using the forgot password feature",
                "payment_failed": "Check card details, ensure sufficient funds, or try alternative payment method",
                "api_errors": "Verify API key validity and check documentation for current rate limits",
                "slow_performance": "Clear cache, check internet connection, or try during off-peak hours"
            },
            "product_features": {
                "api_integration": "REST API with OAuth2 authentication, rate limiting, and comprehensive documentation",
                "billing": "Monthly subscriptions with automated invoicing, multiple payment methods, and prorated charges",
                "support": "24/7 email support with 4-hour response time SLA and dedicated account managers for enterprise"
            }
        }
    def _create_triage_agent(self) -> Agent:
        """Agent that categorizes and routes incoming support requests"""
        return Agent(
            role="Support Triage Specialist",
            goal="Analyze customer queries and route them to the appropriate specialist agent",
            backstory="""You are an experienced customer support triage agent with exceptional pattern
            recognition skills. You can quickly understand the nature of customer issues, determine
            urgency levels, and identify the most appropriate resolution path based on issue complexity,
            customer history, and available resources.""",
            tools=[],
            llm=self.llm_config,
            verbose=True,
            max_iter=3
        )

    def _create_technical_agent(self) -> Agent:
        """Agent that handles technical issues and product questions"""
        return Agent(
            role="Technical Support Engineer",
            goal="Resolve technical issues using deep product knowledge and troubleshooting expertise",
            backstory="""You are a senior technical support engineer with comprehensive understanding
            of the product's architecture, common error patterns, and advanced troubleshooting
            methodologies. You excel at explaining complex technical concepts in simple terms
            and providing step-by-step solutions that customers can easily follow.""",
            tools=[self.KnowledgeBaseTool(self.knowledge_base)],
            llm=self.llm_config,
            verbose=True,
            max_iter=5
        )

    def _create_billing_agent(self) -> Agent:
        """Agent that handles billing, subscriptions, and account management"""
        return Agent(
            role="Billing and Account Specialist",
            goal="Resolve billing issues, explain subscription details, and manage account configurations",
            backstory="""You are a knowledgeable billing specialist with expertise in subscription
            management, payment processing, and account administration. You understand complex
            billing scenarios, can explain charges clearly, and help customers optimize their
            subscription plans based on usage patterns and business needs.""",
            tools=[],
            llm=self.llm_config,
            verbose=True,
            max_iter=4
        )

    def _create_escalation_agent(self) -> Agent:
        """Agent that manages HITL escalations and ensures smooth handoffs"""
        return Agent(
            role="Escalation Coordinator",
            goal="Manage human-in-the-loop escalations and ensure seamless transitions to human agents",
            backstory="""You are an experienced escalation manager who understands when AI assistance
            reaches its limits and human intervention is required. You excel at summarizing complex
            situations, preparing comprehensive handoff documentation, and setting clear expectations
            for both customers and human support agents during transitions.""",
            tools=[],
            llm=self.llm_config,
            verbose=True,
            max_iter=2
        )

**Explanation:** Each agent is carefully crafted with specific roles, goals, and backstories that define their expertise and behavior. The triage agent acts as the entry point, using pattern recognition to categorize issues. Technical agents leverage deep product knowledge, while billing agents specialize in financial matters. The escalation agent serves as a crucial bridge to human support, determining when AI capabilities are insufficient and ensuring smooth transitions. The max_iter parameter controls how many reasoning steps each agent can take, balancing thoroughness with efficiency.
## Implementing Human-in-the-Loop (HITL) Workflow

In [15]:
class SimpleSupportTool(BaseTool):
    name: str = "SimpleSupport"
    description: str = "Basic support tool for common issues"

    def _run(self, query: str) -> str:
        """Provide simple support responses"""
        common_solutions = {
            "login": "Try resetting your password or clearing browser cache",
            "billing": "Check your payment method and subscription status",
            "api": "Verify your API key and check documentation",
            "error": "Check our status page or try again later"
        }

        for key, solution in common_solutions.items():
            if key in query.lower():
                return solution

        return "I'll help you troubleshoot this issue. Can you provide more details?"

In [29]:
import requests
import json
from datetime import datetime
from typing import Dict, List, Any

class CustomerSupportCrew:
    def __init__(self):
        self.api_key = "sk-or-v1-e3ecd714bc83f6ee13f9d49fe97f29c1eb92c8b7987623b7a8db52a6df4ed438a" #remove 'a' at the end to use this api-key.
        self.base_url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "http://localhost:5000",
            "X-Title": "Customer Support System"
        }

        self.knowledge_base = self._load_knowledge_base()
        self.conversation_history = []

    def _load_knowledge_base(self) -> Dict:
        return {
            "login": "Password reset: use 'Forgot Password' on login page",
            "billing": "Payment issues: update payment method in account settings",
            "api": "API errors: regenerate API key and check documentation"
        }

    def _call_openrouter(self, prompt: str, role: str, system_message: str) -> str:
        """Make direct API call to OpenRouter"""
        data = {
            "model": "mistralai/mistral-7b-instruct",
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 300
        }

        try:
            response = requests.post(self.base_url, headers=self.headers, json=data, timeout=30)
            response.raise_for_status()
            result = response.json()
            return result['choices'][0]['message']['content']
        except Exception as e:
            return f"Error: {str(e)}"

    def process_customer_query(self, customer_message: str, customer_id: str = None) -> Dict[str, Any]:
        try:
            # Add to conversation history
            self.conversation_history.append({
                "timestamp": datetime.now().isoformat(),
                "customer_id": customer_id,
                "message": customer_message,
                "agent": "customer"
            })

            # Simple triage logic
            category = self._categorize_query(customer_message)

            if category == "escalation":
                return self._handle_escalation(customer_message, customer_id)
            else:
                return self._handle_standard_query(customer_message, customer_id, category)

        except Exception as e:
            return self._handle_error(e, customer_id)

    def _categorize_query(self, message: str) -> str:
        message_lower = message.lower()
        urgent_keywords = ["urgent", "emergency", "down", "critical", "not working", "broken"]
        billing_keywords = ["bill", "payment", "invoice", "charge", "subscription", "price"]
        technical_keywords = ["error", "bug", "api", "integration", "technical", "code"]

        if any(word in message_lower for word in urgent_keywords):
            return "escalation"
        elif any(word in message_lower for word in billing_keywords):
            return "billing"
        elif any(word in message_lower for word in technical_keywords):
            return "technical"
        else:
            return "general"

    def _handle_escalation(self, message: str, customer_id: str) -> Dict[str, Any]:
        """Handle escalation with direct API call"""
        system_message = """You are an escalation manager. Create brief escalation notes for urgent customer issues that need human intervention."""

        response = self._call_openrouter(
            prompt=f"Create escalation notes for: {message}",
            role="escalation",
            system_message=system_message
        )

        return {
            "status": "escalated",
            "agent_type": "human",
            "response": "I'm escalating this to our support team for immediate assistance.",
            "customer_id": customer_id,
            "timestamp": datetime.now().isoformat(),
            "notes": response
        }

    def _handle_standard_query(self, message: str, customer_id: str, category: str) -> Dict[str, Any]:
        """Handle standard queries with direct API calls"""
        if category == "billing":
            system_message = """You are a billing specialist. Help customers with payment, subscription, and billing issues. Be helpful and professional."""
            role = "billing"
        else:
            system_message = """You are a technical support engineer. Help customers with login, API, and technical issues. Provide step-by-step solutions."""
            role = "technical"

        response = self._call_openrouter(
            prompt=f"Customer issue: {message}",
            role=role,
            system_message=system_message
        )

        return {
            "status": "resolved",
            "agent_type": category,
            "response": response,
            "customer_id": customer_id,
            "timestamp": datetime.now().isoformat()
        }

    def _handle_error(self, error: Exception, customer_id: str) -> Dict[str, Any]:
        return {
            "status": "error",
            "agent_type": "system",
            "response": "I apologize, but I'm experiencing technical difficulties. Please try again or contact support directly.",
            "customer_id": customer_id,
            "timestamp": datetime.now().isoformat(),
            "error": str(error)
        }

**Explanation:** The HITL implementation creates a sophisticated workflow that automatically determines when human intervention is necessary. The system begins with intelligent triage that analyzes issue complexity, urgency, and resolution likelihood. When escalation thresholds are met, the system generates comprehensive handoff packages that preserve context and provide human agents with actionable insights. This approach maintains customer satisfaction by ensuring seamless transitions while optimizing AI utilization for routine inquiries.

## Knowledge Base Tools and Multimodal Capabilities

In [30]:
# Define tools in global scope so they're accessible to all classes
class KnowledgeBaseTool(BaseTool):
    name: str = "KnowledgeBaseLookup"
    description: str = "Search the product knowledge base for solutions and information"
    knowledge_base: Dict[str, Any] = None  # Define as class variable

    def __init__(self, knowledge_base: Dict):
        # Call super().__init__() first, then set the attribute
        super().__init__()
        # Use object.__setattr__ to bypass Pydantic validation for dynamic fields
        object.__setattr__(self, 'knowledge_base', knowledge_base)

    def _run(self, query: str) -> str:
        """Search knowledge base for relevant information"""
        if not self.knowledge_base:
            return "Knowledge base not available"

        results = []
        for category, items in self.knowledge_base.items():
            for key, value in items.items():
                if query.lower() in key.lower() or any(word in value.lower() for word in query.lower().split()):
                    results.append(f"{category}.{key}: {value}")

        return "\n".join(results) if results else "No relevant information found."

class DocumentAnalysisTool(BaseTool):
    name: str = "DocumentAnalyzer"
    description: str = "Analyze uploaded documents, images, or screenshots for additional context"

    def _run(self, file_path: str) -> str:
        """Analyze document content"""
        return f"Analyzed document: {file_path}. Content extraction would happen here in production."

**Explanation:** The knowledge base tools provide agents with access to structured information, enabling consistent and accurate responses. The multimodal document analysis tool demonstrates how CrewAI can integrate with vision capabilities to handle screenshots, error messages, and other visual content that customers often provide. In a production environment, these tools would connect to vector databases, computer vision APIs, and other advanced AI services to enhance agent capabilities.
## Complete SaaS Support Bot Implementation

In [31]:
class SaaSSupportBot:
    def __init__(self):
        # Initialize support crew
        self.support_crew = CustomerSupportCrew()
        self.performance_metrics = {
            "requests_handled": 0,
            "escalations": 0,
            "successful_resolutions": 0,
            "average_response_time": 0.0
        }

    def handle_request(self, user_message: str, user_id: str = "anonymous") -> Dict[str, Any]:
        """Main entry point for handling support requests"""
        start_time = datetime.now()

        try:
            # Process through support crew
            result = self.support_crew.process_customer_query(user_message, user_id)

            # Update metrics
            self._update_metrics(result, start_time)

            return result

        except Exception as e:
            # Handle errors at bot level
            error_result = self._handle_bot_error(e, user_id, start_time)
            return error_result

    def _update_metrics(self, result: Dict, start_time: datetime):
        """Update performance metrics"""
        self.performance_metrics["requests_handled"] += 1

        if result.get("status") == "escalated":
            self.performance_metrics["escalations"] += 1
        elif result.get("status") == "resolved":
            self.performance_metrics["successful_resolutions"] += 1

        # Update average response time
        response_time = (datetime.now() - start_time).total_seconds()
        current_avg = self.performance_metrics["average_response_time"]
        total = self.performance_metrics["requests_handled"]

        if total > 0:
            self.performance_metrics["average_response_time"] = (
                (current_avg * (total - 1) + response_time) / total
            )

    def _handle_bot_error(self, error: Exception, user_id: str, start_time: datetime) -> Dict[str, Any]:
        """Handle errors at the bot level"""
        self.performance_metrics["requests_handled"] += 1

        return {
            "status": "error",
            "agent_type": "system",
            "response": "Our support system is currently experiencing issues. Please contact support directly.",
            "customer_id": user_id,
            "timestamp": datetime.now().isoformat(),
            "error": str(error)
        }

    def get_metrics(self) -> Dict[str, Any]:
        """Get current performance metrics"""
        return self.performance_metrics.copy()

    def reset_metrics(self):
        """Reset performance metrics"""
        self.performance_metrics = {
            "requests_handled": 0,
            "escalations": 0,
            "successful_resolutions": 0,
            "average_response_time": 0.0
        }

**Explanation:** The complete SaaS Support Bot integrates all components into a production-ready system. It handles various request types, processes multimodal attachments, tracks performance metrics, and provides comprehensive reporting. The example demonstrates real-world usage scenarios including technical support, billing inquiries, and urgent escalations. The metrics tracking enables continuous improvement by monitoring key performance indicators like response times and escalation rates.

## Production Deployment Considerations

In [32]:
class ProductionSupportCrew(CustomerSupportCrew):
    """Enhanced version with production-ready features"""

    def __init__(self, database_connection=None, monitoring_hook=None):
        super().__init__()
        self.db = database_connection
        self.monitoring = monitoring_hook
        self.error_rate = 0
        self.consecutive_errors = 0

    def process_customer_query(self, customer_message: str, customer_id: str = None) -> Dict[str, Any]:
        """Production version with enhanced error handling and monitoring"""
        try:
            # Log request for monitoring
            if self.monitoring:
                self.monitoring.log_request(customer_id, len(customer_message))

            result = super().process_customer_query(customer_message, customer_id)

            # Reset error counter on success
            self.consecutive_errors = 0

            # Store in database if available
            if self.db:
                self.db.store_interaction(customer_id, customer_message, result)

            return result

        except Exception as e:
            # Enhanced error handling
            self.consecutive_errors += 1
            self.error_rate = min(1.0, self.consecutive_errors / 10)  # Cap at 100%

            # Critical error threshold
            if self.consecutive_errors > 5:
                if self.monitoring:
                    self.monitoring.alert_critical("High error rate detected")

            # Fallback response
            return {
                "status": "error",
                "agent_type": "system",
                "response": "I apologize, but I'm experiencing technical difficulties. Please try again or contact support directly.",
                "customer_id": customer_id,
                "timestamp": datetime.now().isoformat(),
                "error": str(e)
            }

# Monitoring and alerting integration
class MonitoringSystem:
    def log_request(self, user_id: str, message_length: int):
        print(f"📊 Monitoring: Request from {user_id}, length: {message_length}")

    def alert_critical(self, message: str):
        print(f"🚨 CRITICAL ALERT: {message}")

# Database integration example
class SupportDatabase:
    def store_interaction(self, user_id: str, message: str, result: Dict):
        print(f"💾 Database: Storing interaction for {user_id}")

**Explanation:** The production-ready version incorporates essential enterprise features including comprehensive error handling, monitoring integration, database persistence, and alerting systems. These enhancements ensure reliability, maintainability, and observability in real-world deployments. The system gracefully handles failures, maintains error rate metrics, and provides fallback mechanisms to maintain service availability even during partial outages.

This complete CrewAI implementation demonstrates how to build sophisticated multi-agent systems that can handle complex customer support workflows with production-grade reliability and advanced features like HITL escalation and multimodal processing.

In [33]:
# Test your CrewAI system
def test_crewai_system():
    print("🧪 Testing CrewAI Customer Support System...")

    # Create the support bot
    support_bot = SaaSSupportBot()
    print("✅ Support bot created successfully!")

    # Test cases
    test_messages = [
        ("I can't login to my account", "user_123"),
        ("My payment failed", "user_456"),
        ("URGENT: API is down!", "user_789"),
        ("How do I use the REST API?", "user_999")
    ]

    for i, (message, user_id) in enumerate(test_messages, 1):
        print(f"\n--- Test Case {i} ---")
        print(f"User: {user_id}")
        print(f"Message: {message}")

        # Process the request
        result = support_bot.handle_request(message, user_id)

        print(f"Status: {result.get('status', 'N/A')}")
        print(f"Agent: {result.get('agent_type', 'N/A')}")
        print(f"Response: {result.get('response', 'No response')}")

    # Show metrics
    print(f"\n📊 Final Metrics:")
    metrics = support_bot.get_metrics()
    for key, value in metrics.items():
        print(f"{key}: {value}")

# Run the test
test_crewai_system()

🧪 Testing CrewAI Customer Support System...
✅ Support bot created successfully!

--- Test Case 1 ---
User: user_123
Message: I can't login to my account
Status: resolved
Agent: general
Response:  I'm sorry to hear you're having trouble logging in to your account. Here are some steps you can take to troubleshoot the issue:

1. Check your internet connection: Ensure that you have a stable internet connection. Try refreshing the page or reconnecting to the internet and then try logging in again.

2. Check your login credentials: Make sure you are entering the correct email address and password. If you're unsure of your password, click on the "Forgot Password" link to reset it.

3. Clear your browser cache and cookies: Sometimes, old data stored in your browser can cause login issues. Clearing your browser cache and cookies might help. Here's how to do it for common browsers:

   - Google Chrome: Click on the three dots in the top right corner of the browser, then select "More tools" > "Cl